# Peel vs Slice Visualization

Both decompositions turn `slab region \ reservation prism` into a collection
of convex space-time sets, using the prism's own facets as decision
half-spaces (the ECD step from `fixed_interval.ipynb` Part 2, Step D). They
differ only in whether the remainder shrinks between facets:

- **Peel** (disjoint) — `fixed_interval.ipynb`'s `peel_free_space` /
  `ecd.py`'s `_peel_fixed`. Walk the facets in order; at facet `i` keep
  `mid ∩ {outside facet i}` as a set, then shrink `mid ← mid ∩ {inside
  facet i}`. Every later set lives inside the earlier facets, so sets have
  **pairwise-disjoint interiors**.
- **Slice** (overlapping) — no shrinking. At facet `i` keep
  `region ∩ {outside facet i}` directly. Each set is the full half of the
  slab on the far side of one facet, so neighbouring sets **overlap**.

Note: `ecd.py`'s function named `slice()` actually peels; "slice" here
means only the overlapping variant above.

Both cover exactly `region \ prism` (a point outside a convex obstacle
violates at least one facet), and both produce at most one set per facet.

**Space-time setup** — same as `fixed_interval.ipynb` Part 2: time is
pre-cut into fixed `Delta` slabs, an agent's trajectory is one cubic Bezier
per slab, and its reservation in slab `k` is the inflated spatial hull
extruded over `[k*Delta, (k+1)*Delta]`. The prism's lateral facets have zero
time coefficient, so every set is itself a prism spanning the whole slab:
the sets of slab `k` stack between timesteps `k*Delta` and `(k+1)*Delta`,
and slabs stack on top of each other along `t`.

All 3D plots are `x, y, t` — space is the horizontal plane, time is
vertical. Interactive: drag to rotate, click legend entries to toggle sets,
or use the dropdown to isolate one set.

In [1]:
from stgcs.geometry_utils import *
from stgcs.collision_utils import *

In [2]:
import itertools
import numpy as np
import networkx as nx
import plotly.graph_objects as go
import plotly.colors as pc
from plotly.subplots import make_subplots
from scipy.special import comb
from scipy.spatial import ConvexHull, HalfspaceIntersection

d = 2                          # 2D space
order = 4                      # cubic Bezier (4 control points)
DELTA = 0.5                    # fixed, predetermined period
N_SLABS = 5                    # number of Delta-slices in the horizon
BOX = (0.0, 10.0, 0.0, 10.0)   # xmin, xmax, ymin, ymax
FOOT_R = 0.4                   # square footprint half-width
AREA_EPS = 1e-6
GAP = 0.04 * DELTA             # visual-only gap between stacked slabs, so the layers read as separate

print(f"Delta={DELTA}, N_SLABS={N_SLABS}, horizon=[0,{N_SLABS*DELTA}], box={BOX}, footprint half-width={FOOT_R}")

Delta=0.5, N_SLABS=5, horizon=[0,2.5], box=(0.0, 10.0, 0.0, 10.0), footprint half-width=0.4


Geometry helpers. `inflate_hull_dspace` is `fixed_interval.ipynb` Step C's,
with one addition: facets are sorted by normal angle so that "facet `i`"
(and so "set `i`") points the same way in every slab — without it, the
facet order `ConvexHull` returns can differ slab to slab, and the per-set
colours/toggles below would mean different things in different slabs.

In [3]:
def box_hpoly():
    xmin, xmax, ymin, ymax = BOX
    return HPolyhedron.MakeBox(np.array([xmin, ymin]), np.array([xmax, ymax]))

def footprint_vertices():
    return np.array([[sx * FOOT_R, sy * FOOT_R] for sx in (-1, 1) for sy in (-1, 1)])

def inflate_hull_dspace(control_points_space, footprint_vertices_space):
    raw = (control_points_space[:, None, :] + footprint_vertices_space[None, :, :]).reshape(
        -1, control_points_space.shape[-1])
    h = make_hpolytope(raw).ReduceInequalities()
    A, b = h.A(), h.b()
    idx = np.argsort(np.mod(np.arctan2(A[:, 1], A[:, 0]) + 1e-9, 2 * np.pi))  # canonical facet order
    return HPolyhedron(A[idx], b[idx])

def reservation_prisms(segments):
    # per slab: project control points to space (drop T), inflate by footprint
    return [inflate_hull_dspace(Q[:, :d], footprint_vertices()) for Q in segments]

def polygon_vertices_2d(hpoly2d):
    # HalfspaceIntersection rather than hpoly_to_vrep -- see fixed_interval.ipynb's note on cddlib instability
    A, b = hpoly2d.A(), hpoly2d.b()
    try:
        interior = hpoly2d.ChebyshevCenter()
        V = HalfspaceIntersection(np.hstack([A, -b.reshape(-1, 1)]), interior).intersections
    except Exception:
        V = hpoly_to_vrep(hpoly2d)
    if V is None or len(V) < 3:
        return None
    try:
        V = V[ConvexHull(V).vertices]
    except Exception:
        return None
    c = V.mean(axis=0)
    return V[np.argsort(np.arctan2(V[:, 1] - c[1], V[:, 0] - c[0]))]

def polygon_area(hpoly2d):
    if hpoly2d.IsEmpty():
        return 0.0
    V = polygon_vertices_2d(hpoly2d)
    return 0.0 if V is None else ConvexHull(V).volume

def straight_line_segments(start_xy, goal_xy):
    # One cubic Bezier per slab along a straight line at constant speed -- the
    # exact optimum fixed_interval.ipynb Step B's solve_chain returns for an
    # unobstructed straight crossing, built directly so no solver is needed here.
    a, b = np.asarray(start_xy, float), np.asarray(goal_xy, float)
    segs = []
    for k in range(N_SLABS):
        s = (k + np.arange(order) / (order - 1)) / N_SLABS
        P = a + np.outer(s, b - a)
        T = k * DELTA + np.arange(order) / (order - 1) * DELTA
        segs.append(np.hstack([P, T[:, None]]))
    return segs

print("geometry helpers defined")

geometry helpers defined


Plotting helpers — `fixed_interval.ipynb`'s `prism_mesh` / `box_wireframe` /
`curve_trace` / `make_layout`, generalised so one trace can hold the same
set across every slab (one legend entry toggles it through all of time).

In [4]:
def bezier_eval(Q, num=25):
    deg = Q.shape[0] - 1
    s = np.linspace(0, 1, num)
    B = np.array([comb(deg, i) * s**i * (1 - s)**(deg - i) for i in range(deg + 1)])
    return B.T @ Q

def _prism_geometry(V, t0, t1, offset):
    # vertices + triangles for one prism with footprint V over [t0, t1]
    m = len(V)
    verts = np.vstack([np.hstack([V, np.full((m, 1), t0)]), np.hstack([V, np.full((m, 1), t1)])])
    I, J, K = [], [], []
    for e in range(m):
        e2 = (e + 1) % m
        I += [e, e]; J += [e2, m + e2]; K += [m + e2, m + e]
    for e in range(1, m - 1):
        I += [0, m]; J += [e, m + e]; K += [e + 1, m + e + 1]
    return verts, np.array(I) + offset, np.array(J) + offset, np.array(K) + offset

def prisms_trace(polys, color, opacity=0.3, name="", legendgroup=None, showlegend=True, gap=GAP):
    # polys: list of (hpoly2d, t0, t1) merged into a single Mesh3d
    verts, I, J, K, off = [], [], [], [], 0
    for hpoly, t0, t1 in polys:
        V = polygon_vertices_2d(hpoly)
        if V is None:
            continue
        v, i, j, k = _prism_geometry(V, t0 + gap, t1 - gap, off)
        verts.append(v); I.append(i); J.append(j); K.append(k); off += len(v)
    verts = np.vstack(verts)
    return go.Mesh3d(x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
                     i=np.concatenate(I), j=np.concatenate(J), k=np.concatenate(K),
                     color=color, opacity=opacity, flatshading=True, name=name,
                     legendgroup=legendgroup or name, showlegend=showlegend, hoverinfo="name")

def prisms_edges(polys, color, width=2, name="", legendgroup=None, gap=GAP):
    # wireframe of the same prisms, so overlapping translucent sets stay distinguishable
    xs, ys, zs = [], [], []
    for hpoly, t0, t1 in polys:
        V = polygon_vertices_2d(hpoly)
        if V is None:
            continue
        m = len(V)
        for t in (t0 + gap, t1 - gap):
            for i in range(m + 1):
                xs.append(V[i % m, 0]); ys.append(V[i % m, 1]); zs.append(t)
            xs.append(None); ys.append(None); zs.append(None)
        for v in V:
            xs += [v[0], v[0], None]; ys += [v[1], v[1], None]; zs += [t0 + gap, t1 - gap, None]
    return go.Scatter3d(x=xs, y=ys, z=zs, mode="lines", line=dict(color=color, width=width),
                        name=name, legendgroup=legendgroup or name, showlegend=False, hoverinfo="skip")

def slab_wireframes(color="#bbbbbb"):
    return prisms_edges([(box_hpoly(), k * DELTA, (k + 1) * DELTA) for k in range(N_SLABS)],
                        color, width=2, name="Delta slabs", gap=0.0).update(showlegend=True)

def curve_trace(segments, color, name):
    pts = np.vstack([bezier_eval(Q) for Q in segments])
    return go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode="lines",
                        line=dict(color=color, width=7), name=name)

def reservation_traces(obstacles, color, name, slabs=None):
    slabs = range(len(obstacles)) if slabs is None else slabs
    polys = [(obstacles[k], k * DELTA, (k + 1) * DELTA) for k in slabs]
    return [prisms_trace(polys, color, opacity=0.55, name=name),
            prisms_edges(polys, "black", width=2, name=name)]

def make_layout(title, width=900, height=750):
    return go.Layout(
        title=title,
        scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="t (time)",
                   xaxis=dict(range=[BOX[0], BOX[1]]), yaxis=dict(range=[BOX[2], BOX[3]]),
                   aspectmode="manual", aspectratio=dict(x=1, y=1, z=1.2),
                   camera=dict(eye=dict(x=1.6, y=-1.6, z=0.9))),
        legend=dict(itemsizing="constant"), width=width, height=height,
        margin=dict(l=0, r=0, t=50, b=0),
    )

SET_COLORS = pc.qualitative.Plotly   # 10 colours; set i -> SET_COLORS[i]
print("plot helpers defined")

plot helpers defined


## Scene — one agent's reservation through the fixed-`Delta` slabs

Agent 1 crosses diagonally `(1,1) -> (7,7)` over `N_SLABS` slabs (the same
crossing as `fixed_interval.ipynb` Step B). The diagonal makes each slab's
reservation a hexagonal prism (6 facets) rather than an axis-aligned box,
so the two decompositions look visibly different.

In [5]:
START1, GOAL1 = (1.0, 1.0), (7.0, 7.0)
seg1 = straight_line_segments(START1, GOAL1)
obstacles1 = reservation_prisms(seg1)
regions = [box_hpoly() for _ in range(N_SLABS)]

for k, (Q, obs) in enumerate(zip(seg1, obstacles1)):
    print(f"slab{k}: t=[{Q[0,2]:.2f},{Q[-1,2]:.2f}]  P0={np.round(Q[0,:2],2)} -> P3={np.round(Q[-1,:2],2)}  "
          f"reservation: {obs.A().shape[0]} lateral facets")

fig_scene = go.Figure(layout=make_layout("Agent 1's reservation prisms in the fixed-Delta slabs"))
fig_scene.add_trace(slab_wireframes())
for tr in reservation_traces(obstacles1, "crimson", "agent 1 reservation"):
    fig_scene.add_trace(tr)
fig_scene.add_trace(curve_trace(seg1, "crimson", "agent 1"))
fig_scene

slab0: t=[0.00,0.50]  P0=[1. 1.] -> P3=[2.2 2.2]  reservation: 6 lateral facets
slab1: t=[0.50,1.00]  P0=[2.2 2.2] -> P3=[3.4 3.4]  reservation: 6 lateral facets
slab2: t=[1.00,1.50]  P0=[3.4 3.4] -> P3=[4.6 4.6]  reservation: 6 lateral facets
slab3: t=[1.50,2.00]  P0=[4.6 4.6] -> P3=[5.8 5.8]  reservation: 6 lateral facets
slab4: t=[2.00,2.50]  P0=[5.8 5.8] -> P3=[7. 7.]  reservation: 6 lateral facets


## The two decompositions

`peel_decompose` is `fixed_interval.ipynb`'s `peel_free_space`;
`slice_decompose` is the same loop with the shrink step removed. Each is
applied to every slab against that slab's prism. Sets are returned as
`(facet index, set)` pairs so set `i` is always "the set cut off by facet
`i`".

In [6]:
def outside_halfspace(A, b, i):
    return HPolyhedron(-A[i:i + 1], -b[i:i + 1])

def inside_halfspace(A, b, i):
    return HPolyhedron(A[i:i + 1], b[i:i + 1])

def peel_decompose(region2d, obstacle2d, area_eps=AREA_EPS):
    # disjoint: keep mid ∩ outside_i, then shrink mid to mid ∩ inside_i
    A, b = obstacle2d.A(), obstacle2d.b()
    out, mid = [], region2d
    for i in range(A.shape[0]):
        if mid.IsEmpty():
            break
        piece = mid.Intersection(outside_halfspace(A, b, i))
        if not piece.IsEmpty() and polygon_area(piece) > area_eps:
            out.append((i, piece))
        mid = mid.Intersection(inside_halfspace(A, b, i))
    return out

def slice_decompose(region2d, obstacle2d, area_eps=AREA_EPS):
    # overlapping: keep region ∩ outside_i for every facet, region never shrinks
    A, b = obstacle2d.A(), obstacle2d.b()
    out = []
    for i in range(A.shape[0]):
        piece = region2d.Intersection(outside_halfspace(A, b, i))
        if not piece.IsEmpty() and polygon_area(piece) > area_eps:
            out.append((i, piece))
    return out

peel_per_slab = [peel_decompose(r, o) for r, o in zip(regions, obstacles1)]
slice_per_slab = [slice_decompose(r, o) for r, o in zip(regions, obstacles1)]
for k in range(N_SLABS):
    print(f"slab{k}: peel -> {len(peel_per_slab[k])} sets, slice -> {len(slice_per_slab[k])} sets")

slab0: peel -> 6 sets, slice -> 6 sets
slab1: peel -> 6 sets, slice -> 6 sets
slab2: peel -> 6 sets, slice -> 6 sets
slab3: peel -> 6 sets, slice -> 6 sets
slab4: peel -> 6 sets, slice -> 6 sets


### Sanity checks

On a dense grid over each slab's cross-section: (1) both decompositions
cover exactly the free space (every free point is in ≥1 set, no
prism-interior point is in any set); (2) peel's coverage count is 1 almost
everywhere (only shared boundaries hit 2+), slice's is not. Since every set
is a full-height prism within its slab, the 2D cross-section check is the
whole check.

In [7]:
GRID_N = 300
gx, gy = np.meshgrid(np.linspace(BOX[0], BOX[1], GRID_N), np.linspace(BOX[2], BOX[3], GRID_N))
grid = np.stack([gx.ravel(), gy.ravel()], axis=1)

def contains(hpoly2d, pts, tol=1e-9):
    with np.errstate(all="ignore"):  # macOS Accelerate emits spurious matmul warnings
        return np.all(pts @ hpoly2d.A().T <= hpoly2d.b() + tol, axis=1)

def strictly_inside(hpoly2d, pts, tol=1e-6):
    with np.errstate(all="ignore"):
        return np.all(pts @ hpoly2d.A().T < hpoly2d.b() - tol, axis=1)

def coverage_count(sets, pts):
    return np.sum([contains(s, pts) for s in sets], axis=0)

def check(name, sets_per_slab, obstacles_per_slab):
    # obstacles_per_slab[k]: list of obstacles carved from slab k
    max_cov, multi = 0, []
    for k, sets in enumerate(sets_per_slab):
        cov = coverage_count([s for _, s in sets], grid)
        free = np.all([~contains(o, grid) for o in obstacles_per_slab[k]], axis=0)
        inside = np.any([strictly_inside(o, grid) for o in obstacles_per_slab[k]], axis=0)
        assert np.all(cov[free] >= 1), f"{name} slab{k}: some free point uncovered"
        assert np.all(cov[inside] == 0), f"{name} slab{k}: a set enters a reservation"
        max_cov = max(max_cov, cov.max()); multi.append(np.mean(cov[free] > 1))
    print(f"{name:5s}: covers all free space in every slab ✓ | max coverage {max_cov} "
          f"| free points in >1 set: {100 * np.mean(multi):.1f}%")

for name, sps in [("peel", peel_per_slab), ("slice", slice_per_slab)]:
    check(name, sps, [[o] for o in obstacles1])

peel : covers all free space in every slab ✓ | max coverage 1 | free points in >1 set: 0.0%
slice: covers all free space in every slab ✓ | max coverage 3 | free points in >1 set: 97.2%


## Visualization 1 — Peel: space-time decomposed into disjoint sets

Each set is a prism over its slab's time window, coloured by the facet that
cut it off (same colour = same facet across slabs). Within a slab the
prisms tile the free space with no overlap — they meet only along shared
faces — and the slabs stack along `t` (the thin gaps between slabs are
visual only).

Use the dropdown (or click legend entries) to isolate one set: peel's sets
shrink as the facet index grows, because each one only gets what the
earlier facets left behind.

In [8]:
def decomposition_figure(sets_per_slab, title, opacity, extra_traces=()):
    fig = go.Figure(layout=make_layout(title))
    fig.add_trace(slab_wireframes())
    n_facets = 1 + max(i for sets in sets_per_slab for i, _ in sets)
    set_trace_idx = {}
    for i in range(n_facets):
        polys = [(s, k * DELTA, (k + 1) * DELTA) for k, sets in enumerate(sets_per_slab) for fi, s in sets if fi == i]
        if not polys:
            continue
        color = SET_COLORS[i % len(SET_COLORS)]
        start = len(fig.data)
        fig.add_trace(prisms_trace(polys, color, opacity=opacity, name=f"set {i} (facet {i})", legendgroup=f"set{i}"))
        fig.add_trace(prisms_edges(polys, color, width=3, name=f"set {i} (facet {i})", legendgroup=f"set{i}"))
        set_trace_idx[i] = (start, start + 1)
    for tr in extra_traces:
        fig.add_trace(tr)

    # dropdown: all sets, or one set at a time (slabs, reservation, trajectory stay visible)
    n = len(fig.data)
    all_set = {j for pair in set_trace_idx.values() for j in pair}
    buttons = [dict(label="all sets", method="update", args=[{"visible": [True] * n}])]
    for i, pair in set_trace_idx.items():
        vis = [(j not in all_set) or (j in pair) for j in range(n)]
        buttons.append(dict(label=f"set {i} only", method="update", args=[{"visible": vis}]))
    fig.update_layout(updatemenus=[dict(buttons=buttons, direction="down", x=0.0, xanchor="left", y=1.0, yanchor="top")])
    return fig

def agent1_traces():
    return reservation_traces(obstacles1, "crimson", "agent 1 reservation") + [curve_trace(seg1, "crimson", "agent 1")]

fig_peel = decomposition_figure(peel_per_slab, "Visualization 1 — PEEL: disjoint space-time sets", opacity=0.35,
                                extra_traces=agent1_traces())
fig_peel

## Visualization 2 — Slice: space-time decomposed into overlapping sets

Same slabs, same prisms, same colouring — but each set is now the whole
slab on the far side of one facet. Inside every slab the prisms overlap
(colours mix wherever two or three sets share volume), and each set is far
larger than its peel counterpart. Isolate a set with the dropdown to see a
full half-slab extruded over `[k*Delta, (k+1)*Delta]`, stacked through
every slab.

In [9]:
fig_slice = decomposition_figure(slice_per_slab, "Visualization 2 — SLICE: overlapping space-time sets", opacity=0.15,
                                 extra_traces=agent1_traces())
fig_slice

### One slab, side by side

Zoomed to a single slab — the space between two timesteps `k*Delta` and
`(k+1)*Delta` — with peel on the left and slice on the right. Here the
slab's time window is split into one thin layer per set, stacked in facet
order, so every set is visible in full instead of hidden inside the others
(the layering is visual only; every set really spans the whole window).
Looking down the stack: peel's layers each cover a different patch of the
plane and together tile it once; slice's layers each cover a large
half-plane, so the same `(x, y)` shows up in several layers — that repeated
footprint is the overlap.

In [10]:
K_SHOW = 2      # which slab to zoom into

def exploded_slab_traces(sets, k, scene, opacity, show_legend):
    t0 = k * DELTA
    top = t0 + DELTA
    h = DELTA / len(sets)            # one thin layer per set inside [t0, t0 + Delta]
    traces = []
    for rank, (i, s) in enumerate(sets):
        color = SET_COLORS[i % len(SET_COLORS)]
        polys = [(s, t0 + rank * h + 0.08 * h, t0 + (rank + 1) * h - 0.08 * h)]
        traces.append(prisms_trace(polys, color, opacity=opacity, name=f"set {i}", legendgroup=f"set{i}",
                                   showlegend=show_legend, gap=0.0))
        traces.append(prisms_edges(polys, color, width=3, name=f"set {i}", legendgroup=f"set{i}", gap=0.0))
    # reservation spans the whole slab, the hole every layer wraps around
    res = [(obstacles1[k], t0, top)]
    traces.append(prisms_trace(res, "crimson", opacity=0.55, name="agent 1 reservation", showlegend=show_legend, gap=0.0))
    traces.append(prisms_edges(res, "black", width=2, name="agent 1 reservation", gap=0.0))
    pts = bezier_eval(seg1[k])
    traces.append(go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode="lines",
                               line=dict(color="crimson", width=7), name="agent 1", showlegend=show_legend))
    return [tr.update(scene=scene) for tr in traces]

fig_slab = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]],
                         subplot_titles=(f"PEEL — slab {K_SHOW}: {len(peel_per_slab[K_SHOW])} disjoint sets",
                                         f"SLICE — slab {K_SHOW}: {len(slice_per_slab[K_SHOW])} overlapping sets"),
                         horizontal_spacing=0.02)
for tr in exploded_slab_traces(peel_per_slab[K_SHOW], K_SHOW, "scene", opacity=0.6, show_legend=True):
    fig_slab.add_trace(tr)
for tr in exploded_slab_traces(slice_per_slab[K_SHOW], K_SHOW, "scene2", opacity=0.6, show_legend=False):
    fig_slab.add_trace(tr)
scene_kw = dict(xaxis_title="x", yaxis_title="y", zaxis_title="t (one layer per set)",
                xaxis=dict(range=[BOX[0], BOX[1]]), yaxis=dict(range=[BOX[2], BOX[3]]),
                aspectmode="manual", aspectratio=dict(x=1, y=1, z=1.0),
                camera=dict(eye=dict(x=1.7, y=-1.7, z=1.1)))
fig_slab.update_layout(scene=scene_kw, scene2=scene_kw, width=1300, height=650,
                       title=f"Slab {K_SHOW}: t in [{K_SHOW*DELTA}, {(K_SHOW+1)*DELTA}], one layer per set",
                       margin=dict(l=30, r=0, t=70, b=0), legend=dict(itemsizing="constant"))
fig_slab

## Quantitative comparison

Totals over all slabs (space-time volume = cross-section area × `Delta`):

- **sum of set volumes / free volume** — 1.0 means no redundancy; above
  1.0 is how much space-time is counted more than once.
- **mean set volume** — how much room each GCS vertex gives the solver.
- **intra-slab edges** — pairs of sets in the same slab that intersect
  (closure). For peel this means sharing a face; for slice, genuine overlap.
- **inter-slab edges** — pairs of sets in consecutive slabs whose
  cross-sections intersect, i.e. the edges the ST-GCS needs between
  timesteps (same test as `fixed_interval.ipynb`'s `build_adjacency_local`).
- **hops start→goal** — fewest sets a path from `(1, 7)` at `t=0` to
  `(7, 1)` at `t=N_SLABS*Delta` must visit, crossing agent 1's corridor (as
  agent 2 does in `fixed_interval.ipynb`). Every slab must be visited, so
  `N_SLABS` is the floor; anything above that is switching sets inside a
  slab. (Graph reachability only — a necessary, not sufficient, condition
  for a feasible trajectory.)

In [11]:
def st_graph(sets_per_slab):
    G = nx.Graph()
    for k, sets in enumerate(sets_per_slab):
        G.add_nodes_from((k, j) for j in range(len(sets)))
    intra = inter = 0
    for k, sets in enumerate(sets_per_slab):
        for a, b in itertools.combinations(range(len(sets)), 2):
            if not sets[a][1].Intersection(sets[b][1]).IsEmpty():
                G.add_edge((k, a), (k, b)); intra += 1
        if k + 1 < len(sets_per_slab):
            for a, (_, sa) in enumerate(sets):
                for b, (_, sb) in enumerate(sets_per_slab[k + 1]):
                    if not sa.Intersection(sb).IsEmpty():
                        G.add_edge((k, a), (k + 1, b)); inter += 1
    return G, intra, inter

def hops(sets_per_slab, G, start, goal):
    s = [(0, j) for j, (_, p) in enumerate(sets_per_slab[0]) if p.PointInSet(np.array(start))]
    g = [(len(sets_per_slab) - 1, j) for j, (_, p) in enumerate(sets_per_slab[-1]) if p.PointInSet(np.array(goal))]
    best = min((nx.shortest_path_length(G, a, c) for a in s for c in g if nx.has_path(G, a, c)), default=None)
    return None if best is None else best + 1

START2, GOAL2 = (1.0, 7.0), (7.0, 1.0)

def summarize(name, sets_per_slab, free_area_per_slab):
    G, intra, inter = st_graph(sets_per_slab)
    vols = [polygon_area(s) * DELTA for sets in sets_per_slab for _, s in sets]
    return dict(method=name, n_sets=len(vols), sum_vol_over_free=sum(vols) / (sum(free_area_per_slab) * DELTA),
                mean_vol=np.mean(vols), min_vol=np.min(vols),
                intra_slab_edges=intra, inter_slab_edges=inter, hops_start_goal=hops(sets_per_slab, G, START2, GOAL2))

def print_table(rows):
    print(f"{'':22s}" + "".join(f"{r['method']:>12s}" for r in rows))
    for k in list(rows[0].keys())[1:]:
        print(f"{k:22s}" + "".join(f"{r[k]:>12.3f}" if isinstance(r[k], float) else f"{str(r[k]):>12s}" for r in rows))

free_area1 = [polygon_area(r) - polygon_area(o) for r, o in zip(regions, obstacles1)]
print_table([summarize("peel", peel_per_slab, free_area1), summarize("slice", slice_per_slab, free_area1)])

                              peel       slice
n_sets                          30          30
sum_vol_over_free            1.000       2.511
mean_vol                     8.120      20.387
min_vol                      0.330       3.000
intra_slab_edges                35          58
inter_slab_edges                68         118
hops_start_goal                  6           5


## Composition — a second agent's reservation

The difference compounds once a second agent's prisms are carved in.
Agent 2 crosses `(9, 3) -> (2, 9)`, chosen so its reservation never
intersects agent 1's in any slab. Peel is applied set-by-set
(`fixed_interval.ipynb`'s `carve_fragments`): sets that don't touch the new
prism are kept as-is, others are peeled. Slice does the same with
`slice_decompose`: every existing set the new prism hits is replaced by its
slices — and since slice sets overlap, one patch of space-time can be
re-sliced once per set covering it.

Sets are no longer tied to one facet, so the legend groups by slab instead:
click a slab to toggle that whole layer between its two timesteps.

In [12]:
def carve(sets_per_slab, obstacles, decompose):
    out = []
    for k, sets in enumerate(sets_per_slab):
        new = []
        for _, s in sets:
            if s.Intersection(obstacles[k]).IsEmpty():
                new.append((None, s))
            else:
                new.extend((None, p) for _, p in decompose(s, obstacles[k]))
        out.append(new)
    return out

seg2 = straight_line_segments((9.0, 3.0), (2.0, 9.0))
obstacles2 = reservation_prisms(seg2)
assert all(o1.Intersection(o2).IsEmpty() for o1, o2 in zip(obstacles1, obstacles2)), "agents' reservations collide"

peel2 = carve(peel_per_slab, obstacles2, peel_decompose)
slice2 = carve(slice_per_slab, obstacles2, slice_decompose)
both = [[o1, o2] for o1, o2 in zip(obstacles1, obstacles2)]
for name, sps in [("peel", peel2), ("slice", slice2)]:
    check(name, sps, both)
    print(f"       sets per slab: {[len(s) for s in sps]}")

def composed_figure(sets_per_slab, title, opacity):
    fig = go.Figure(layout=make_layout(title))
    fig.add_trace(slab_wireframes())
    pal = pc.qualitative.Dark24
    for k, sets in enumerate(sets_per_slab):
        for j, (_, s) in enumerate(sets):
            polys = [(s, k * DELTA, (k + 1) * DELTA)]
            color = pal[(j + 7 * k) % len(pal)]
            fig.add_trace(prisms_trace(polys, color, opacity=opacity, name=f"slab {k} ({len(sets)} sets)",
                                       legendgroup=f"slab{k}", showlegend=(j == 0)))
            fig.add_trace(prisms_edges(polys, color, width=2, name=f"slab {k}", legendgroup=f"slab{k}"))
    for tr in reservation_traces(obstacles1, "crimson", "agent 1 reservation"):
        fig.add_trace(tr)
    for tr in reservation_traces(obstacles2, "royalblue", "agent 2 reservation"):
        fig.add_trace(tr)
    fig.add_trace(curve_trace(seg1, "crimson", "agent 1"))
    fig.add_trace(curve_trace(seg2, "royalblue", "agent 2"))
    return fig

fig_peel2 = composed_figure(peel2, f"PEEL after 2 reservations — {sum(map(len, peel2))} disjoint sets", opacity=0.3)
fig_peel2

peel : covers all free space in every slab ✓ | max coverage 1 | free points in >1 set: 0.0%
       sets per slab: [11, 11, 13, 13, 13]
slice: covers all free space in every slab ✓ | max coverage 9 | free points in >1 set: 98.6%
       sets per slab: [20, 20, 24, 19, 20]


In [13]:
fig_slice2 = composed_figure(slice2, f"SLICE after 2 reservations — {sum(map(len, slice2))} overlapping sets", opacity=0.07)
fig_slice2

In [14]:
free_area2 = [polygon_area(r) - polygon_area(o1) - polygon_area(o2) for r, o1, o2 in zip(regions, obstacles1, obstacles2)]
print_table([summarize("peel", peel2, free_area2), summarize("slice", slice2, free_area2)])

                              peel       slice
n_sets                          61         103
sum_vol_over_free            1.000       4.711
mean_vol                     3.882      10.831
min_vol                      0.009       1.210
intra_slab_edges                95         597
inter_slab_edges               154        1001
hops_start_goal                  6           5


## Part 2 — how peel vs slice affect planning

Three agents plan one after another on an H-shaped corridor, using
`fixed_interval.ipynb` Part 2's pipeline unchanged: for each agent, build
the fragment adjacency graph between consecutive slabs
(`build_adjacency_local`), enumerate candidate fragment sequences with a
capped graph search (`candidate_paths_local`), solve each one as a convex
chain (`solve_chain`), keep the cheapest. The agent's reservation prisms
are then carved into the fragments (`carve_fragments`) before the next
agent plans — once with peel, once with slice.

**Two maps, same H-shaped corridor.** Two blocks (top-middle and
bottom-middle) leave an H of free space: a left column, a right column and
a crossbar, split into 3 convex sets two ways:

- **disjoint map** — the crossbar runs only *between* the columns: the
  three sets touch along the columns' inner edges but never overlap.
- **overlapping map** — the crossbar runs the full width, so it overlaps
  both columns where it crosses them.

As in `fixed_interval.ipynb`, each slab's whole Bezier segment must stay
inside the one fragment chosen for that slab, so the fragment shapes decide
which routes exist. This is a visualization study: 3 agents, one run.

In [15]:
from pydrake.all import (
    GraphOfConvexSets as GCS, GraphOfConvexSetsOptions, LinearEqualityConstraint, LinearConstraint,
    LorentzConeConstraint, QuadraticCost, Binding, Constraint, Cost, MathematicalProgram, Solve,
)
import time

N_PLAN = 8                     # slabs in the planning horizon: [0, N_PLAN*Delta]
VLIMIT = 8.0
block = d + 1                  # [x, y, T] per control point
n = order * block

# two blocks leave an H-shaped free corridor: columns x<3 and x>7, crossbar 3.5<y<6.5
STATIC_OBS = [(3.0, 7.0, 0.0, 3.5), (3.0, 7.0, 6.5, 10.0)]

AGENTS = [((1.5, 1.0), (8.5, 9.0)),    # agent 0: bottom-left -> top-right
          ((8.5, 1.0), (1.5, 9.0)),    # agent 1: bottom-right -> top-left
          ((1.5, 9.0), (8.5, 1.0))]    # agent 2: top-left -> bottom-right
AGENT_COLORS = ["crimson", "royalblue", "seagreen"]

def rect_hpoly(r):
    return HPolyhedron.MakeBox(np.array([r[0], r[2]]), np.array([r[1], r[3]]))

# agent centres stay one footprint away from the blocks and the walls
C_OBS = [(r[0] - FOOT_R, r[1] + FOOT_R, r[2] - FOOT_R, r[3] + FOOT_R) for r in STATIC_OBS]
C_BOX = (BOX[0] + FOOT_R, BOX[1] - FOOT_R, BOX[2] + FOOT_R, BOX[3] - FOOT_R)

xl, xr = C_OBS[0][0], C_OBS[0][1]          # inner edges of the two columns
yb, yt = C_OBS[0][3], C_OBS[1][2]          # crossbar bottom / top
left_col = (C_BOX[0], xl, C_BOX[2], C_BOX[3])
right_col = (xr, C_BOX[1], C_BOX[2], C_BOX[3])
BASE_RECTS = {"disjoint map": [left_col, right_col, (xl, xr, yb, yt)],                 # crossbar between columns
              "overlapping map": [left_col, right_col, (C_BOX[0], C_BOX[1], yb, yt)]}  # crossbar full width
BASE_MAPS = {k: [rect_hpoly(r) for r in v] for k, v in BASE_RECTS.items()}
for k, v in BASE_RECTS.items():
    print(f"{k:16s}: " + ", ".join(f"[{r[0]:.2f},{r[1]:.2f}]x[{r[2]:.2f},{r[3]:.2f}]" for r in v))

disjoint map    : [0.40,2.60]x[0.40,9.60], [7.40,9.60]x[0.40,9.60], [2.60,7.40]x[3.90,6.10]
overlapping map : [0.40,2.60]x[0.40,9.60], [7.40,9.60]x[0.40,9.60], [0.40,9.60]x[3.90,6.10]


In [16]:
def static_obstacle_traces(t_top, scene=None, showlegend=True):
    polys = [(rect_hpoly(r), 0.0, t_top) for r in STATIC_OBS]
    trs = [prisms_trace(polys, "#555555", opacity=0.6, name="static obstacles", showlegend=showlegend, gap=0.0),
           prisms_edges(polys, "black", width=2, name="static obstacles", gap=0.0)]
    return [t.update(scene=scene) for t in trs] if scene else trs

MAP_COLORS = ["#9467bd", "#ff7f0e", "#17becf"]
MAP_NAMES = ["left column", "right column", "crossbar"]
fig_maps = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]], horizontal_spacing=0.02,
                         subplot_titles=list(BASE_MAPS))
for col, (name, sets) in enumerate(BASE_MAPS.items()):
    scene = "scene" if col == 0 else "scene2"
    h = DELTA / len(sets)                       # one thin layer per set, like Part 1's slab zoom
    for j, s in enumerate(sets):
        polys = [(s, j * h + 0.1 * h, (j + 1) * h - 0.1 * h)]
        fig_maps.add_trace(prisms_trace(polys, MAP_COLORS[j], opacity=0.75, name=MAP_NAMES[j],
                                        showlegend=(col == 0), legendgroup=f"m{j}", gap=0.0).update(scene=scene))
        fig_maps.add_trace(prisms_edges(polys, MAP_COLORS[j], width=2, name=MAP_NAMES[j], legendgroup=f"m{j}", gap=0.0).update(scene=scene))
    for tr in static_obstacle_traces(DELTA, scene=scene, showlegend=(col == 0)):
        fig_maps.add_trace(tr)
map_scene = dict(xaxis_title="x", yaxis_title="y", zaxis_title="one layer per set",
                 xaxis=dict(range=[BOX[0], BOX[1]]), yaxis=dict(range=[BOX[2], BOX[3]]),
                 aspectmode="manual", aspectratio=dict(x=1, y=1, z=0.5), camera=dict(eye=dict(x=1.4, y=-1.6, z=1.2)))
fig_maps.update_layout(scene=map_scene, scene2=map_scene, width=1300, height=600, margin=dict(l=30, r=0, t=70, b=0),
                       title="The two H-corridor maps (one slab, one layer per set)")
fig_maps

### Reused from `fixed_interval.ipynb`

Copied verbatim: `add_region_template`, `add_continuity`,
`solve_single_slab`, `solve_chain` (Step A), `build_adjacency_local`,
`candidate_paths_local` (Step E) and `carve_fragments` (Step F). The one
change: `carve_fragments` takes a `decompose` argument so the same carving
loop can slice instead of peel.

In [17]:
def p_slice(i):
    off = i * block
    return slice(off, off + d)

def t_index(i):
    return i * block + d

def add_region_template(gcs, region2d, t0, t1, name):
    """ One GCS vertex = one fixed-Delta cubic Bezier segment confined to
        region2d x [t0,t1] (Step 3/4's corrected mechanism: a *relative*
        per-vertex duration equality, not a per-visit absolute lock). """
    st_hpoly = region2d.CartesianProduct(HPolyhedron.MakeBox([t0], [t1]))
    v = gcs.AddVertex(st_hpoly.CartesianPower(order), name)

    # exact relative duration: T_{order-1} - T_0 == (t1 - t0)
    A_delta = np.zeros((1, n))
    A_delta[0, t_index(0)] = -1.0
    A_delta[0, t_index(order - 1)] = 1.0
    v.AddConstraint(Binding[Constraint](
        LinearEqualityConstraint(A_delta, np.array([t1 - t0])), v.x()))

    # interior monotonicity (free spacing, (a-i))
    A_mono = np.zeros((order - 1, n))
    for i in range(order - 1):
        A_mono[i, t_index(i)] = 1.0
        A_mono[i, t_index(i + 1)] = -1.0
    v.AddConstraint(Binding[Constraint](
        LinearConstraint(A_mono, -np.inf * np.ones(order - 1), np.full(order - 1, -1e-6)),
        v.x()))

    # velocity SOC per span
    for i in range(order - 1):
        A_soc = np.zeros((d + 1, n))
        A_soc[0, t_index(i + 1)] = VLIMIT
        A_soc[0, t_index(i)] = -VLIMIT
        A_soc[1:, p_slice(i + 1)] = np.eye(d)
        A_soc[1:, p_slice(i)] = -np.eye(d)
        v.AddConstraint(Binding[Constraint](LorentzConeConstraint(A_soc, np.zeros(d + 1)), v.x()))

    # energy regularization (replaces the old duration-minimizing cost --
    # duration is now fixed by the equality above, not minimized)
    for i in range(order - 1):
        A_diff = np.zeros((d, n))
        A_diff[:, p_slice(i + 1)] = np.eye(d)
        A_diff[:, p_slice(i)] = -np.eye(d)
        H = 2 * A_diff.T @ A_diff
        H = 0.5 * (H + H.T) + 1e-9 * np.eye(n)
        v.AddCost(Binding[Cost](QuadraticCost(H, np.zeros(n), 0.0), v.x()))

    return v

def add_continuity(e):
    """ C0 (position+time) and C1 (velocity) continuity across an edge. """
    A_c0 = np.zeros((block, 2 * n))
    A_c0[:, (order - 1) * block: order * block] = np.eye(block)
    A_c0[:, n: n + block] = -np.eye(block)
    e.AddConstraint(Binding[Constraint](
        LinearEqualityConstraint(A_c0, np.zeros(block)), np.append(e.xu(), e.xv())))

    A_c1 = np.zeros((block, 2 * n))
    A_c1[:, (order - 1) * block: order * block] = np.eye(block)
    A_c1[:, (order - 2) * block: (order - 1) * block] = -np.eye(block)
    A_c1[:, n: n + block] = np.eye(block)
    A_c1[:, n + block: n + 2 * block] = -np.eye(block)
    e.AddConstraint(Binding[Constraint](
        LinearEqualityConstraint(A_c1, np.zeros(block)), np.append(e.xu(), e.xv())))

def solve_single_slab(region2d, start_xy, goal_xy, t0=0.0):
    """ Fallback for a 1-slab mission -- `GCS.SolveConvexRestriction([])`
        with no edges runs a no-op, empty program (verified directly against
        Drake: without an edge, no vertex is ever "activated"). Rebuilds the
        same per-vertex template as `add_region_template` on a plain
        `MathematicalProgram` instead of a GCS vertex -- there's no
        continuity to enforce with only one segment, so nothing else from
        `solve_chain` is needed. """
    prog = MathematicalProgram()
    x = prog.NewContinuousVariables(n, "x")
    A2, b2 = region2d.A(), region2d.b()
    for i in range(order):
        prog.AddLinearConstraint(A2, -np.inf * np.ones_like(b2), b2, x[p_slice(i)])

    A_delta = np.zeros((1, n))
    A_delta[0, t_index(0)] = -1.0
    A_delta[0, t_index(order - 1)] = 1.0
    prog.AddLinearEqualityConstraint(A_delta, np.array([DELTA]), x)

    A_mono = np.zeros((order - 1, n))
    for i in range(order - 1):
        A_mono[i, t_index(i)] = 1.0
        A_mono[i, t_index(i + 1)] = -1.0
    prog.AddLinearConstraint(A_mono, -np.inf * np.ones(order - 1), np.full(order - 1, -1e-6), x)

    for i in range(order - 1):
        A_soc = np.zeros((d + 1, n))
        A_soc[0, t_index(i + 1)] = VLIMIT
        A_soc[0, t_index(i)] = -VLIMIT
        A_soc[1:, p_slice(i + 1)] = np.eye(d)
        A_soc[1:, p_slice(i)] = -np.eye(d)
        prog.AddLorentzConeConstraint(A_soc, np.zeros(d + 1), x)

    for i in range(order - 1):
        A_diff = np.zeros((d, n))
        A_diff[:, p_slice(i + 1)] = np.eye(d)
        A_diff[:, p_slice(i)] = -np.eye(d)
        H = 2 * A_diff.T @ A_diff
        H = 0.5 * (H + H.T) + 1e-9 * np.eye(n)
        prog.AddQuadraticCost(H, np.zeros(n), x)

    prog.AddLinearEqualityConstraint(np.eye(d), np.array(start_xy), x[0:d])
    prog.AddLinearEqualityConstraint(x[d] == t0)
    prog.AddLinearEqualityConstraint(
        np.eye(d), np.array(goal_xy), x[(order - 1) * block: (order - 1) * block + d])

    result = Solve(prog)
    if not result.is_success():
        return None
    xval = result.GetSolution(x)
    Q = np.array([np.hstack([xval[p_slice(i)], xval[t_index(i)]]) for i in range(order)])
    return [Q], result.get_optimal_cost()

def solve_chain(region2d_per_slab, start_xy, goal_xy):
    """ One vertex per slab (region2d_per_slab[k] used as-is, no
        fragmentation), chained sequentially, source/goal pinned in space
        only. Returns (segments, cost) or None if infeasible. """
    if len(region2d_per_slab) == 1:
        return solve_single_slab(region2d_per_slab[0], start_xy, goal_xy)

    gcs = GCS()
    verts = []
    for k, region2d in enumerate(region2d_per_slab):
        t0, t1 = k * DELTA, (k + 1) * DELTA
        verts.append(add_region_template(gcs, region2d, t0, t1, f"slab{k}"))
    edges = []
    for k in range(len(verts) - 1):
        e = gcs.AddEdge(verts[k], verts[k + 1], f"(slab{k}, slab{k+1})")
        add_continuity(e)
        edges.append(e)

    edges[0].AddConstraint(Binding[Constraint](
        LinearEqualityConstraint(np.eye(d), np.array(start_xy)), edges[0].xu()[0:d]))
    edges[0].AddConstraint(Binding[Constraint](
        LinearEqualityConstraint(np.eye(1), np.array([0.0])), edges[0].xu()[d:d + 1]))
    edges[-1].AddConstraint(Binding[Constraint](
        LinearEqualityConstraint(np.eye(d), np.array(goal_xy)),
        edges[-1].xv()[(order - 1) * block: (order - 1) * block + d]))

    options = GraphOfConvexSetsOptions()
    res = gcs.SolveConvexRestriction(edges, options)
    if not res.is_success():
        return None

    own_vars = [edges[0].xu()] + [e.xv() for e in edges]
    segments = [np.array([np.hstack([res.GetSolution(ov)[p_slice(i)], res.GetSolution(ov)[t_index(i)]])
                           for i in range(order)]) for ov in own_vars]
    return segments, res.get_optimal_cost()

def build_adjacency_local(frags_per_slab):
    """ Same construction as Part 3's `build_adjacency` -- necessary, not
        sufficient, condition for a feasible segment between two fragments. """
    edges = []
    for k in range(len(frags_per_slab) - 1):
        e = []
        for i, fi in enumerate(frags_per_slab[k]):
            for j, fj in enumerate(frags_per_slab[k + 1]):
                if not fi.Intersection(fj).IsEmpty():
                    e.append((i, j))
        edges.append(e)
    return edges

def candidate_paths_local(frags_per_slab, adjacency, start_xy, goal_xy, max_candidates=200):
    """ Same construction as Part 3's `candidate_paths_to_depth` -- a real graph
        search (networkx DFS, capped), not the Cartesian-product enumeration
        this cell used to run. That enumeration is `prod(len(f) for f in
        fragments_per_slab)` candidates regardless of whether they're even
        adjacent -- fine at N_SLABS=4 (1296), but it scales as (fragment
        count)^N_SLABS, so it stops being usable almost immediately as either
        grows (e.g. 6^10 = 60,466,176 at N_SLABS=10). Graph search only
        follows edges that actually exist.

        `nx.has_path` gates `all_simple_paths` the same way Part 3's
        `candidate_paths_to_depth` does -- a genuine "no path" case otherwise forces
        an exhaustive search (measured directly at 26.9s on one real
        instance) instead of the ~0.0004s a plain reachability check takes
        to reach the identical answer. """
    start_frags = [i for i, f in enumerate(frags_per_slab[0]) if f.PointInSet(np.array(start_xy))]
    goal_frags = [j for j, f in enumerate(frags_per_slab[-1]) if f.PointInSet(np.array(goal_xy))]
    if not start_frags or not goal_frags:
        return []
    G = nx.DiGraph()
    for k in range(len(frags_per_slab)):
        for idx in range(len(frags_per_slab[k])):
            G.add_node((k, idx))
    for k, edges_k in enumerate(adjacency):
        for (i, j) in edges_k:
            G.add_edge((k, i), (k + 1, j))
    out = []
    for sf in start_frags:
        for gf in goal_frags:
            source, target = (0, sf), (len(frags_per_slab) - 1, gf)
            try:
                if not nx.has_path(G, source, target):
                    continue
                for path in itertools.islice(nx.all_simple_paths(G, source, target), max_candidates):
                    out.append([idx for (_, idx) in path])
                    if len(out) >= max_candidates:
                        return out
            except (nx.NetworkXNoPath, nx.NodeNotFound):
                continue
    return out

def carve_fragments(frags_per_slab, obstacles, decompose=None):
    """ Peel every existing fragment (already carved by prior agents)
        against one more agent's per-slab reservation prism -- the same
        `peel_free_space` operation Step D ran against the single starting
        region, now composed across agents. Skips fragments that don't
        intersect this slab's obstacle at all: `peel_free_space` would still
        return them correctly (region "\\" obstacle reduces to region when
        disjoint), but only if the peeling loop happens to hit a facet that
        trivially separates them -- two disjoint convex polytopes aren't
        always separated by one of the obstacle's own facet hyperplanes, so
        without this check a non-overlapping fragment can get needlessly
        split into several convex pieces carrying identical free space,
        inflating the fragment count that candidate_paths_local's graph
        search has to cope with. """
    new_frags_per_slab = []
    for k, frags in enumerate(frags_per_slab):
        new_frags = []
        for frag in frags:
            if frag.Intersection(obstacles[k]).IsEmpty():
                new_frags.append(frag)
            else:
                new_frags.extend((decompose or peel_free_space)(frag, obstacles[k]))
        new_frags_per_slab.append(new_frags)
    return new_frags_per_slab

# the two carving rules, as fragment-list functions for carve_fragments
def peel_free_space(region2d, obstacle2d):
    return [s for _, s in peel_decompose(region2d, obstacle2d)]

def slice_free_space(region2d, obstacle2d):
    return [s for _, s in slice_decompose(region2d, obstacle2d)]

print("fixed_interval.ipynb planning helpers loaded")

fixed_interval.ipynb planning helpers loaded


### Sequential planning: 2 maps × 2 carving rules

Same loop as `fixed_interval.ipynb` Steps E–F, per agent: adjacency →
candidate fragment paths → `solve_chain` each → keep the cheapest →
carve its reservation prisms (`reservation_prisms`, Part 1) into the
fragments.

In [18]:
MAX_CANDIDATES = 100

def plan_sequential(base_sets, decompose):
    frags = [list(base_sets) for _ in range(N_PLAN)]
    log = []
    for idx, (start, goal) in enumerate(AGENTS):
        t0 = time.perf_counter()
        adjacency = build_adjacency_local(frags)
        paths = candidate_paths_local(frags, adjacency, start, goal, max_candidates=MAX_CANDIDATES)
        best, n_feasible = None, 0
        for path in paths:
            result = solve_chain([frags[k][j] for k, j in enumerate(path)], start, goal)
            if result is None:
                continue
            n_feasible += 1
            if best is None or result[1] < best[1]:
                best = (path, result[1], result[0])
        info = dict(agent=idx, n_frags=sum(map(len, frags)), n_edges=sum(map(len, adjacency)),
                    n_candidates=len(paths), n_feasible=n_feasible, success=best is not None,
                    seconds=time.perf_counter() - t0)
        entry = dict(info=info, graph_sets=frags, segs=None, used_sets=None)
        if best is not None:
            path, info["cost"], segs = best
            entry.update(segs=segs, used_sets=[frags[k][j] for k, j in enumerate(path)])
            frags = carve_fragments(frags, reservation_prisms(segs), decompose)
        entry["carved_sets"] = frags      # fragments right after this agent's reservation is carved
        log.append(entry)
    return log

RUNS = {}
for map_name, base in BASE_MAPS.items():
    for method, fn in [("peel", peel_free_space), ("slice", slice_free_space)]:
        RUNS[(map_name, method)] = plan_sequential(base, fn)

print(f"{'map':16s} {'rule':5s} {'agent':>5s} {'ok':>4s} {'fragments':>9s} {'edges':>6s} {'candidates':>10s} "
      f"{'feasible':>8s} {'cost':>6s} {'sec':>5s}")
for (map_name, method), log in RUNS.items():
    for r in log:
        i = r["info"]
        print(f"{map_name:16s} {method:5s} {i['agent']:5d} {'yes' if i['success'] else 'NO':>4s} {i['n_frags']:9d} "
              f"{i['n_edges']:6d} {i['n_candidates']:10d} {i['n_feasible']:8d} {i.get('cost', float('nan')):6.2f} {i['seconds']:5.1f}")

map              rule  agent   ok fragments  edges candidates feasible   cost   sec
disjoint map     peel      0  yes        24     49        100       33   5.69   1.0
disjoint map     peel      1  yes        82    211        100       99   5.98   3.0
disjoint map     peel      2   NO       133    362          0        0    nan   5.7
disjoint map     slice     0  yes        24     49        100       33   5.69   1.0
disjoint map     slice     1  yes        91    471        100       70   7.58   3.5
disjoint map     slice     2  yes       206   1821        100       91   7.59  16.1
overlapping map  peel      0  yes        24     49        100       33   5.69   1.1
overlapping map  peel      1  yes        82    261        100      100   5.98   3.1
overlapping map  peel      2   NO       137    462          0        0    nan   6.2
overlapping map  slice     0  yes        24     49        100       33   5.69   1.1
overlapping map  slice     1  yes        92    510        100       60   7.5

### Visualization 3 — planning on the disjoint map: peel vs slice

Static blocks are grey columns through the whole horizon. Each agent's
trajectory is drawn with the fragments its chosen path ran through (one per
slab, in the agent's colour) — the corridor `solve_chain` had to keep the
whole segment inside. Toggle an agent's reservation in the legend to see
the prisms later agents were carved around, or "fragments seen by agent 2"
for the whole graph the last agent searched.

In [19]:
def planning_figure(map_name):
    fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]], horizontal_spacing=0.02,
                        subplot_titles=[f"{m.upper()}: {sum(r['info']['success'] for r in RUNS[(map_name, m)])}"
                                        f"/{len(AGENTS)} agents planned" for m in ("peel", "slice")])
    T = N_PLAN * DELTA
    shown = set()                      # legend entries already added (agent 2 may only exist in one panel)
    def once(key):
        new = key not in shown; shown.add(key); return new
    for col, method in enumerate(("peel", "slice")):
        scene, first = ("scene", True) if col == 0 else ("scene2", False)
        log = RUNS[(map_name, method)]
        for tr in static_obstacle_traces(T, scene=scene, showlegend=first):
            fig.add_trace(tr)
        polys = [(s, k * DELTA, (k + 1) * DELTA) for k, sets in enumerate(log[-1]["graph_sets"]) for s in sets]
        fig.add_trace(prisms_edges(polys, "#999999", width=1, name="fragments seen by agent 2")
                      .update(scene=scene, visible="legendonly", showlegend=first))
        for idx, r in enumerate(log):
            if r["segs"] is None:
                continue
            color, name = AGENT_COLORS[idx], f"agent {idx}"
            used = [(s, k * DELTA, (k + 1) * DELTA) for k, s in enumerate(r["used_sets"])]
            fig.add_trace(prisms_trace(used, color, opacity=0.13, name=f"{name} fragments used",
                                       legendgroup=f"{name}c", showlegend=once(f"{name}c")).update(scene=scene))
            fig.add_trace(prisms_edges(used, color, width=2, name=f"{name} fragments used",
                                       legendgroup=f"{name}c").update(scene=scene))
            res = [(o, k * DELTA, (k + 1) * DELTA) for k, o in enumerate(reservation_prisms(r["segs"]))]
            fig.add_trace(prisms_trace(res, color, opacity=0.5, name=f"{name} reservation", legendgroup=f"{name}r",
                                       showlegend=once(f"{name}r")).update(scene=scene, visible="legendonly"))
            fig.add_trace(curve_trace(r["segs"], color, name).update(scene=scene, legendgroup=name, showlegend=once(name)))
    sc = dict(xaxis_title="x", yaxis_title="y", zaxis_title="t (time)",
              xaxis=dict(range=[BOX[0], BOX[1]]), yaxis=dict(range=[BOX[2], BOX[3]]),
              aspectmode="manual", aspectratio=dict(x=1, y=1, z=1.2), camera=dict(eye=dict(x=0.9, y=-1.5, z=1.7)))
    fig.update_layout(scene=sc, scene2=sc, width=1400, height=720, margin=dict(l=0, r=0, t=70, b=0),
                      title=f"Planning on the {map_name}: peel vs slice", legend=dict(itemsizing="constant"))
    return fig

fig_plan_disjoint = planning_figure("disjoint map")
fig_plan_disjoint

### Visualization 4 — planning on the overlapping map: peel vs slice

Same agents, same blocks, starting from the overlapping decomposition.

In [20]:
fig_plan_overlap = planning_figure("overlapping map")
fig_plan_overlap

### What the planning run shows

Read alongside the table above:

- **Peel cuts the graph apart.** On both maps agent 2 gets *zero*
  candidate paths with peel. Its start and goal are each covered by a
  fragment, but no forward chain of pairwise-intersecting fragments links
  them: once agents 0 and 1 are carved in, peel's disjoint pieces meet only
  along thin shared faces, and somewhere along the horizon the chain from
  the left column to the right one breaks. Slice's fragments cover the
  same free space but overlap, so consecutive slabs keep enough
  intersecting pairs for a chain to survive — all 3 agents plan.
- **Slice pays in graph size.** By agent 2 the slice graph has roughly
  1.5× the fragments and 4–5× the edges of the peel graph, and that agent
  takes about 3× longer (more `IsEmpty` adjacency checks and more
  `solve_chain` calls that succeed).
- **Costs aren't comparable across rules here.** `candidate_paths_local`
  keeps the first `MAX_CANDIDATES` paths in DFS order, not the best ones;
  the slice graph has far more paths, so the cap samples a smaller share of
  them (e.g. agent 1 costs 7.58 with slice vs 5.98 with peel, though every
  peel corridor lies inside a slice corridor).
- **Disjoint vs overlapping map matters less than the carving rule** on
  this small H: both maps give the same success pattern. The overlapping
  map mainly adds more edges from the first carve on.

### Visualization 5 — two consecutive slabs after each agent's reservation

One figure per agent, taken right after that agent's reservation is
carved in: rows are the two maps, columns are peel vs slice. Only two
consecutive slabs are shown — the pair where carving this agent created
the most new fragments — so the effect of one reservation is readable.

Inside each slab every fragment gets its own thin layer (visual only; each
really spans the whole slab, as in Part 1's slab zoom). The agent's
reservation prism (its colour) and trajectory run through both slabs;
earlier agents' reservations are black wireframes. Peel's layers tile the
free space around the prism with no repeats; slice's layers each cover a
large half of the slab, so the same patch shows up in several layers.
Panels are blank where that agent found no path (nothing was reserved).

In [21]:
FRAG_COLORS = pc.qualitative.Light24
COMBOS = [(m, r) for m in BASE_MAPS for r in ("peel", "slice")]

def pick_slab_pair(idx):
    # the consecutive pair where carving agent idx added the most fragments (over all runs where it planned)
    gain = np.zeros(N_PLAN)
    for key in COMBOS:
        e = RUNS[key][idx]
        if e["segs"] is not None:
            gain += np.array([len(a) - len(b) for a, b in zip(e["carved_sets"], e["graph_sets"])])
    return int(np.argmax(gain[:-1] + gain[1:]))

def slab_pair_traces(idx, key, k, scene):
    log, entry = RUNS[key], RUNS[key][idx]
    traces = []
    for slab in (k, k + 1):
        frags, t0 = entry["carved_sets"][slab], slab * DELTA
        h = DELTA / len(frags)
        for j, f in enumerate(frags):
            polys = [(f, t0 + (j + 0.12) * h, t0 + (j + 0.88) * h)]
            color = FRAG_COLORS[j % len(FRAG_COLORS)]
            traces.append(prisms_trace(polys, color, opacity=0.7, name=f"slab {slab} fragment {j}", showlegend=False, gap=0.0))
            traces.append(prisms_edges(polys, color, width=1.5, name=f"slab {slab} fragment {j}", gap=0.0))
        res = [(reservation_prisms(entry["segs"])[slab], t0, t0 + DELTA)]
        traces.append(prisms_trace(res, AGENT_COLORS[idx], opacity=0.35, name=f"agent {idx} reservation",
                                   showlegend=False, gap=0.0))
        traces.append(prisms_edges(res, AGENT_COLORS[idx], width=3, name=f"agent {idx} reservation", gap=0.0))
        for prev in log[:idx]:
            if prev["segs"] is not None:
                traces.append(prisms_edges([(reservation_prisms(prev["segs"])[slab], t0, t0 + DELTA)], "black",
                                           width=2, name="earlier reservation", gap=0.0))
    traces.append(curve_trace(entry["segs"][k:k + 2], AGENT_COLORS[idx], f"agent {idx}").update(showlegend=False))
    traces += static_obstacle_traces((k + 2) * DELTA, showlegend=False)
    for tr in traces:
        if tr.z is not None:   # clip the full-height static obstacles to the two slabs
            tr.z = [None if z is None else min(max(z, k * DELTA), (k + 2) * DELTA) for z in tr.z]
    return [tr.update(scene=scene) for tr in traces]

def slab_pair_figure(idx):
    k = pick_slab_pair(idx)
    titles = []
    for key in COMBOS:
        e = RUNS[key][idx]
        if e["segs"] is None:
            titles.append(f"{key[0]} / {key[1].upper()}: agent {idx} found no path")
        else:
            counts = [f"slab {s}: {len(e['graph_sets'][s])}→{len(e['carved_sets'][s])}" for s in (k, k + 1)]
            titles.append(f"{key[0]} / {key[1].upper()} — " + ", ".join(counts))
    fig = make_subplots(rows=2, cols=2, specs=[[{"type": "scene"}] * 2] * 2, subplot_titles=titles,
                        horizontal_spacing=0.01, vertical_spacing=0.05)
    for n_scene, key in enumerate(COMBOS, start=1):
        if RUNS[key][idx]["segs"] is None:
            continue
        for tr in slab_pair_traces(idx, key, k, "scene" if n_scene == 1 else f"scene{n_scene}"):
            fig.add_trace(tr)
    sc = dict(xaxis_title="x", yaxis_title="y", zaxis_title="t (one layer per fragment)",
              xaxis=dict(range=[BOX[0], BOX[1]]), yaxis=dict(range=[BOX[2], BOX[3]]),
              zaxis=dict(range=[k * DELTA, (k + 2) * DELTA]),
              aspectmode="manual", aspectratio=dict(x=1, y=1, z=0.9), camera=dict(eye=dict(x=1.1, y=-1.5, z=1.3)))
    fig.update_layout(**{("scene" if i == 1 else f"scene{i}"): sc for i in range(1, 5)},
                      width=1300, height=1250, margin=dict(l=0, r=0, t=90, b=0),
                      title=f"Agent {idx}: fragments in slabs {k} and {k+1} "
                            f"(t in [{k*DELTA}, {(k+2)*DELTA}]) after carving its reservation")
    return fig

fig_slabs_agent0 = slab_pair_figure(0)
fig_slabs_agent0

In [22]:
fig_slabs_agent1 = slab_pair_figure(1)
fig_slabs_agent1

In [23]:
fig_slabs_agent2 = slab_pair_figure(2)
fig_slabs_agent2

## Part 3 — a decision point inside the contact face

In the H corridor every set switch happened at a block corner: a segment
can't turn a corner inside one rectangle, and the energy cost pulls the
path taut against the corner, so the switching joint lands on an edge even
on the overlapping map. Here is an instance where the overlap actually gets
used.

**Map:** two blocks (top-left, bottom-right) leave a free space that is the
union of two overlapping squares, `A = [0.4, 5.5]²` and `B = [4.5, 9.6]²`
(agent-centre C-space), overlapping in `[4.5, 5.5]²`. One agent goes
`(1, 1) -> (9, 9)` over 8 slabs.

- **overlapping map** — `{A, B}`. The straight diagonal at constant speed is
  feasible and minimum-energy; its joints sit at `(1+k, 1+k)`. Joint 4,
  `(5, 5)` at `t = 2.0`, is the only joint in both squares, so the A → B
  switch happens there — **strictly inside the contact face**
  `(A ∩ B) × {t = 2.0}`, half a unit from every edge.
- **disjoint map** — same free space split into `A`, `B1 = [5.5, 9.6] x [4.5, 9.6]`
  and `B2 = [4.5, 5.5] x [5.5, 9.6]`. `A` and `B1` only share the edge
  `x = 5.5`, so the switch joint is pinned to that edge at a slab boundary
  and the path has to bend to meet it.

Same planner as Part 2 (`candidate_paths_local` + `solve_chain`, no other
agents, no carving); the cap is raised so every candidate is tried.

In [24]:
X_BLOCKS = [(0.0, 4.1, 5.9, 10.0), (5.9, 10.0, 0.0, 4.1)]      # static blocks, before inflation
SQ_A, SQ_B = (0.4, 5.5, 0.4, 5.5), (4.5, 9.6, 4.5, 9.6)        # C-space free squares (blocks grown by FOOT_R)
X_RECTS = {"disjoint map": [SQ_A, (5.5, 9.6, 4.5, 9.6), (4.5, 5.5, 5.5, 9.6)],
           "overlapping map": [SQ_A, SQ_B]}
X_NAMES = {"disjoint map": ["A", "B1", "B2"], "overlapping map": ["A", "B"]}
X_START, X_GOAL = (1.0, 1.0), (9.0, 9.0)

# sanity: both decompositions cover the same C-space free area
x_free = np.zeros(len(grid), bool)
for r in X_RECTS["overlapping map"]:
    x_free |= contains(rect_hpoly(r), grid)
for name, rects in X_RECTS.items():
    cov = coverage_count([rect_hpoly(r) for r in rects], grid)
    assert np.array_equal(cov >= 1, x_free)
    print(f"{name:16s}: {len(rects)} sets, same free space ✓, max coverage {cov.max()}")

def rect_intersection(a, b):
    return (max(a[0], b[0]), min(a[1], b[1]), max(a[2], b[2]), min(a[3], b[3]))

X_RUNS = {}
for name, rects in X_RECTS.items():
    frags = [[rect_hpoly(r) for r in rects] for _ in range(N_PLAN)]
    paths = candidate_paths_local(frags, build_adjacency_local(frags), X_START, X_GOAL, max_candidates=5000)
    best = None
    for path in paths:
        result = solve_chain([frags[k][j] for k, j in enumerate(path)], X_START, X_GOAL)
        if result is not None and (best is None or result[1] < best[1]):
            best = (path, result[1], result[0])
    path, cost, segs = best
    X_RUNS[name] = dict(path=path, cost=cost, segs=segs)
    print(f"\n{name}: {len(paths)} candidate paths, best cost {cost:.3f}, sets per slab: "
          f"{[X_NAMES[name][j] for j in path]}")
    for k in range(N_PLAN - 1):
        p, T = segs[k][-1, :2], segs[k][-1, 2]
        ra, rb = rects[path[k]], rects[path[k + 1]]
        face = rect_intersection(ra, rb)
        slack = min(p[0] - face[0], face[1] - p[0], p[1] - face[2], face[3] - p[1])
        tag = "  <- set switch" if path[k] != path[k + 1] else ""
        print(f"  joint {k}->{k+1}: t={T:.3f}  p=({p[0]:.3f}, {p[1]:.3f})  distance to edge of contact face: "
              f"{max(slack, 0):.3f}{tag}")

disjoint map    : 3 sets, same free space ✓, max coverage 1
overlapping map : 2 sets, same free space ✓, max coverage 2



disjoint map: 729 candidate paths, best cost 5.379, sets per slab: ['A', 'A', 'A', 'A', 'B1', 'B1', 'B1', 'B1']
  joint 0->1: t=0.500  p=(2.136, 2.000)  distance to edge of contact face: 1.600
  joint 1->2: t=1.000  p=(3.273, 3.000)  distance to edge of contact face: 2.227
  joint 2->3: t=1.500  p=(4.409, 4.000)  distance to edge of contact face: 1.091
  joint 3->4: t=2.000  p=(5.500, 5.000)  distance to edge of contact face: 0.000  <- set switch
  joint 4->5: t=2.500  p=(6.409, 6.000)  distance to edge of contact face: 0.909
  joint 5->6: t=3.000  p=(7.273, 7.000)  distance to edge of contact face: 1.773
  joint 6->7: t=3.500  p=(8.136, 8.000)  distance to edge of contact face: 1.464



overlapping map: 64 candidate paths, best cost 5.333, sets per slab: ['A', 'A', 'A', 'A', 'B', 'B', 'B', 'B']
  joint 0->1: t=0.500  p=(2.000, 2.000)  distance to edge of contact face: 1.600
  joint 1->2: t=1.000  p=(3.000, 3.000)  distance to edge of contact face: 2.500
  joint 2->3: t=1.500  p=(4.000, 4.000)  distance to edge of contact face: 1.500
  joint 3->4: t=2.000  p=(5.000, 5.000)  distance to edge of contact face: 0.500  <- set switch
  joint 4->5: t=2.500  p=(6.000, 6.000)  distance to edge of contact face: 1.500
  joint 5->6: t=3.000  p=(7.000, 7.000)  distance to edge of contact face: 2.500
  joint 6->7: t=3.500  p=(8.000, 8.000)  distance to edge of contact face: 1.600


### Visualization 6 — where the switch happens

Each panel shows the blocks, the set used in every slab (coloured by set),
the trajectory with every joint as a small dot, and — at each **set
switch** — the contact face `fragment_k ∩ fragment_{k+1}` drawn flat at
`t = kΔ` (a yellow patch, or a yellow line when the sets only touch) with
the decision point as a black diamond. On the overlapping map the diamond
sits in the middle of a square patch; on the disjoint map it sits on a line.

In [25]:
SET_FILL = {"A": "#9467bd", "B": "#ff7f0e", "B1": "#ff7f0e", "B2": "#17becf"}

def flat_rect_trace(r, z, scene):
    x0, x1, y0, y1 = r
    if x1 - x0 < 1e-9 or y1 - y0 < 1e-9:      # sets only touch: the contact face is a segment
        return go.Scatter3d(x=[x0, x1], y=[y0, y1], z=[z, z], mode="lines", line=dict(color="gold", width=12),
                            name="contact face", showlegend=False, scene=scene)
    return go.Mesh3d(x=[x0, x1, x1, x0], y=[y0, y0, y1, y1], z=[z] * 4, i=[0, 0], j=[1, 2], k=[2, 3],
                     color="gold", opacity=0.9, name="contact face", showlegend=False, scene=scene)

fig_switch = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]], horizontal_spacing=0.02,
                           subplot_titles=[f"{n}: sets {[X_NAMES[n][j] for j in X_RUNS[n]['path']]}, cost {X_RUNS[n]['cost']:.2f}"
                                           for n in X_RECTS])
shown = set()
for col, name in enumerate(X_RECTS, start=1):
    scene = "scene" if col == 1 else "scene2"
    run, rects = X_RUNS[name], X_RECTS[name]
    blocks = [(rect_hpoly(r), 0.0, N_PLAN * DELTA) for r in X_BLOCKS]
    fig_switch.add_trace(prisms_trace(blocks, "#555555", opacity=0.3, name="static blocks", gap=0.0,
                                      showlegend=col == 1).update(scene=scene))
    for k, j in enumerate(run["path"]):
        set_name = X_NAMES[name][j]
        polys = [(rect_hpoly(rects[j]), k * DELTA, (k + 1) * DELTA)]
        fig_switch.add_trace(prisms_trace(polys, SET_FILL[set_name], opacity=0.12, name=f"set {set_name}",
                                          legendgroup=set_name, showlegend=set_name not in shown).update(scene=scene))
        fig_switch.add_trace(prisms_edges(polys, SET_FILL[set_name], width=2, name=f"set {set_name}",
                                          legendgroup=set_name).update(scene=scene))
        shown.add(set_name)
    fig_switch.add_trace(curve_trace(run["segs"], "crimson", "trajectory").update(scene=scene, showlegend=col == 1))
    J = np.array([Q[-1] for Q in run["segs"][:-1]])
    fig_switch.add_trace(go.Scatter3d(x=J[:, 0], y=J[:, 1], z=J[:, 2], mode="markers", marker=dict(size=3, color="crimson"),
                                      name="joints (t = kΔ)", showlegend=col == 1, scene=scene))
    for k in range(N_PLAN - 1):
        a, b = run["path"][k], run["path"][k + 1]
        if a == b:
            continue
        fig_switch.add_trace(flat_rect_trace(rect_intersection(rects[a], rects[b]), (k + 1) * DELTA, scene))
        p = run["segs"][k][-1]
        fig_switch.add_trace(go.Scatter3d(x=[p[0]], y=[p[1]], z=[p[2]], mode="markers",
                                          marker=dict(size=7, color="black", symbol="diamond"),
                                          name="decision point (set switch)", showlegend=(col == 1), scene=scene))
sc = dict(xaxis_title="x", yaxis_title="y", zaxis_title="t (time)",
          xaxis=dict(range=[BOX[0], BOX[1]]), yaxis=dict(range=[BOX[2], BOX[3]]),
          aspectmode="manual", aspectratio=dict(x=1, y=1, z=1.2), camera=dict(eye=dict(x=0.25, y=-2.0, z=1.0)))
fig_switch.update_layout(scene=sc, scene2=sc, width=1400, height=720, margin=dict(l=0, r=0, t=70, b=0),
                         title="Where the set switch happens: disjoint vs overlapping decomposition",
                         legend=dict(itemsizing="constant"))
fig_switch

## Takeaways

- **Peel → disjoint.** Set volumes sum to exactly the free space-time; sets
  meet only along shared faces within a slab. Set shapes depend on facet
  order (the `fixed`/`greedy` choice in `ecd.py`), and later sets are
  squeezed into the corners the earlier ones left behind.
- **Slice → overlapping.** Same number of sets per reservation, but each is
  a full half-slab, so space-time is counted several times over. GCS sees
  larger, more-connected vertices: a trajectory can switch sets anywhere in
  an overlap, and consecutive slabs have many more edges between them.
- **Under composition** slice's redundancy compounds — every overlapping
  set touching the new prism is sliced independently, so set count and
  edge count grow much faster than peel's. Dropping sets contained in
  another (or merging) would be the natural next step if slice were used
  in the pipeline.